# Exploratory Data Analysis (EDA) — OULAD

**DSP391m · Group 1 · FPT University** · Topic: *Time-Aware Explainable ML for Early At-Risk Student Detection (OULAD)*

> **Executable EDA**: every analysis is run through the tested `src/eda/eda.py` module, so the figures & tables never drift from the code, and the numbers printed here are the ones used on the slides. We use `master_raw` (t=100%) for univariate/bivariate/correlation, and the six checkpoint datasets for the time-aware analysis.



## Executive summary

On the master table (**32,593** records, at-risk rate **52.8%**), five findings shape the modelling plan:

1. **Behaviour dominates demographics.** Engagement & assessment features separate the classes strongly (Cohen's *d* up to **2.55**; all **19/19** numeric features significant at *q* < 0.05); demographic association is weak (Cramér's *V* ≤ **0.15**).
2. **Clickstream is strongly right-skewed** (skew up to ~35) → motivates the `log1p` transform.
3. **`days_since_last_activity` is bimodal** — the structural signature of disengagement.
4. **No leakage:** no feature correlates ≥ 0.95 with the label; two engagement pairs are multicollinear (|r| ≥ 0.8) and flagged for SHAP attribution.
5. **Signal is early:** `n_days_active` and `days_since_last_activity` reach a large effect (*d* ≥ 0.8) by **10%** of course length, scores by **20%** — grounding the early-intervention goal (RQ1).

## 0. Objectives & statistical method

**What:** understand data quality and shape, find features that separate at-risk students, check for leakage, and locate when the signal emerges over time.

**Why:** EDA is not decoration — each chart drives a decision (*feature selection* or *proving no leakage*).

**Method:** Mann–Whitney U (non-parametric, fits the skewed features) + Benjamini–Hochberg correction & **Cohen's d** for numeric features; **chi-square + Cramér's V** for categorical features; **Pearson/Spearman** for multivariate structure. Figures follow the team chart standard (`src/eda/plot_style.py`, 300 dpi, colour-blind safe).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
TABLES = ROOT / 'reports' / 'tables'
FIGS = ROOT / 'reports' / 'figures'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
from src.eda import eda
master = eda.load_master()
checkpoints = eda.load_checkpoints()
print('master:', master.shape,
      '| checkpoints stacked:', None if checkpoints is None else checkpoints.shape)

## 1. Dataset profile (structure-first)

An analyst's first pass: for every column — its type, completeness, cardinality and (for numeric features) distribution shape. This single table replaces guesswork and flags exactly what cleaning must address (the few columns with gaps and the strongly skewed clickstream features).

In [ ]:
num_cols = master.select_dtypes("number").columns
profile = pd.DataFrame({
    "dtype": master.dtypes.astype(str),
    "n_missing": master.isnull().sum(),
    "pct_missing": (master.isnull().mean() * 100).round(2),
    "n_unique": master.nunique(),
    "skew": master[num_cols].skew().round(2),
}).sort_values("pct_missing", ascending=False)
key = ["code_module", "code_presentation", "id_student"]
print(f"{len(master):,} rows x {master.shape[1]} columns  |  numeric = {len(num_cols)}  |  "
      f"duplicate composite keys = {master.duplicated(key).sum()}  |  "
      f"at-risk rate = {master['at_risk'].mean():.1%}")
display(profile)

## 2. Data quality

Only three columns contain missing values: `date_unregistration` (structurally absent for students who never withdraw — not a feature), `imd_band` (1,111 ≈ 3.4%; filled `Unknown`), and `date_registration` (45 ≈ 0.14%; train-median imputed). After cleaning there is **0 NaN** in any feature column.

In [ ]:
_ = eda.data_quality(master)
display(pd.read_csv(TABLES / 'data_quality_profile.csv', index_col=0))
display(Image(filename=str(FIGS / 'quality_missingness.png')))

## 3. Univariate analysis

**What:** histogram + KDE and boxplots per numeric feature.

**Why:** most clickstream features are **strongly right-skewed** — the heavy tails are *real* learning behaviour, so we use `log1p` rather than deleting rows.

**Result:** `clicks_resource` skew ≈ **34.7**, kurtosis ≈ **2,125** → evidence for `log1p`; `days_since_last_activity` is **bimodal**. Non-normal ⇒ later steps use **non-parametric** tests.

In [ ]:
_ = eda.univariate(master)
num = pd.read_csv(TABLES / 'univariate_numeric.csv', index_col=0)
display(num[['mean','std','min','50%','max','skew','kurtosis']])
display(Image(filename=str(FIGS / 'univariate_hist_kde.png')))

In [ ]:
display(Image(filename=str(FIGS / 'univariate_boxplots.png')))

**Categorical.** 83% of students hold A-Level or below; Post-Graduate / No-Formal are rare (<1.1% each) — hence one-hot with `handle_unknown=ignore` rather than dropping rare levels.

In [ ]:
display(Image(filename=str(FIGS / 'univariate_categorical_freq.png')))

## 4. Target distribution & class imbalance

The observed at-risk rate is **52.8%** (imbalance ratio 1.12): a *slight majority*, not a severe imbalance — the real figure, not 68/32. Because a missed at-risk student is the costly error, PR-AUC and recall on the at-risk class are the headline metrics. **Withdrawn** is the single largest class, which matters for the time-aware analysis.

In [ ]:
_ = eda.target_distribution(master)
counts = master['final_result'].value_counts().rename_axis('final_result').reset_index(name='count')
counts['pct'] = (counts['count'] / len(master) * 100).round(1)
counts['label'] = counts['final_result'].isin(['Fail','Withdrawn']).map({True:'at-risk (1)', False:'not-at-risk (0)'})
display(counts)
display(Image(filename=str(FIGS / 'target_distribution.png')))

## 5. Bivariate — numeric features vs the target

**What:** for each numeric feature, **Cohen's d** + a Mann–Whitney U test (BH-corrected).

**Why:** at n≈32k almost every p-value is tiny → **rank by effect size**, not p.

**Result:** strongest are behavioural/performance — `days_since_last_activity` (*d*=**2.55**, 171 vs 14 days), `n_assessments_submitted` (2.05, 2.3 vs 8.6), `weighted_score_to_date` (1.96). **All 19/19 features significant**.

In [ ]:
_ = eda.numeric_vs_target(master)
tb = pd.read_csv(TABLES / 'bivariate_numeric_tests.csv').sort_values('cohens_d', ascending=False)
display(tb[['feature','group','mean_not_at_risk','mean_at_risk','cohens_d','p_adj_bh','significant_(q<0.05)']].head(10))
print(f"Significant features (q<0.05): {int(tb['significant_(q<0.05)'].sum())}/{len(tb)}")
display(Image(filename=str(FIGS / 'bivariate_effect_sizes.png')))

In [ ]:
display(Image(filename=str(FIGS / 'bivariate_top_boxplots.png')))

## 6. Bivariate — categorical features vs the target (Cramér's V)

Chi-square tests are significant (large *n*), but **Cramér's V** effect sizes are small: `highest_education` (0.15) and `imd_band` (0.15) lead, while `gender` (0.02) is negligible. Demographics carry limited standalone signal and are best kept for **fairness analysis** (watch deprivation and education, not gender), not prediction.

In [ ]:
_ = eda.categorical_vs_target(master)
display(pd.read_csv(TABLES / 'bivariate_categorical_tests.csv'))
display(Image(filename=str(FIGS / 'bivariate_categorical_rate.png')))

## 7. Multivariate — correlation, multicollinearity, leakage

**What:** Pearson + Spearman matrices; correlation with the label; leakage check.

**Why:** leakage is a **mandatory** check — a feature with |r| ≥ 0.95 vs the label is suspicious.

**Result:** top |r| with the label is `days_since_last_activity` **+0.78** (agreeing with Cohen's d → two independent methods, same conclusion); **no feature |r| ≥ 0.95** ⇒ no leakage; two multicollinear pairs ≥ 0.8 (flagged for RQ2/SHAP).

In [ ]:
_ = eda.correlation(master)
print('Strong pairs (|r| >= 0.6):')
display(pd.read_csv(TABLES / 'correlation_strong_pairs.csv'))
print('Correlation with the target (top |r|):')
display(pd.read_csv(TABLES / 'correlation_with_target.csv', index_col=0))
display(Image(filename=str(FIGS / 'corr_pearson.png')))

In [ ]:
display(Image(filename=str(FIGS / 'corr_with_target.png')))

## 8. Time-aware analysis — when does the signal emerge? (RQ1)

**What:** measure |Cohen's d| per feature at six checkpoints 10%→100%.

**Why:** the analysis specific to a time-aware problem — find the earliest reliable checkpoint for prediction (RQ1).

**Result:** discrimination grows steadily; first to reach *d* ≥ 0.8: `n_days_active` **and** `days_since_last_activity` from **10%**, scores/submissions from **20%** → early prediction is feasible from **~10–20%** of course length.

> *Consistency note:* after fixing the no-activity fill for `days_since_last_activity`, its *d* at t=100% (per-checkpoint table) equals **2.55**, matching the value computed on `master_raw` in §5.

In [ ]:
ta = eda.time_aware(checkpoints)
display(pd.read_csv(TABLES / 'discrimination_by_checkpoint.csv', index_col=0).round(3))
print('Earliest checkpoint reaching d>=0.8:', ta['earliest_checkpoint_d_ge_0.8'])
display(Image(filename=str(FIGS / 'time_discrimination_curve.png')))

In [ ]:
display(Image(filename=str(FIGS / 'time_mean_trajectory.png')))

## 9. The Withdrawn early-warning signal (Step-0 Option A)

Withdrawal produces a **genuine signal**, not noise: median days-since-last-activity rises from **11** (not-at-risk) → **116** (Fail) → **233** (Withdrawn); median total clicks collapse from ~1,425 to ~89.

**Honest caveat:** because withdrawn students are near-inactive, they are *trivially* separable at **late** checkpoints → late-checkpoint recall/PR-AUC can look optimistic; the genuine test of the model is at the **early** checkpoints.

In [ ]:
_ = eda.withdrawn_analysis(master)
import numpy as np
m = master.copy()
m['status'] = np.where(m['final_result'].eq('Withdrawn'), 'Withdrawn',
                       np.where(m['at_risk'].eq(1), 'Fail', 'Not-at-risk'))
summary = m.groupby('status').agg(
    median_days_idle=('days_since_last_activity', 'median'),
    median_total_clicks=('total_clicks', 'median'),
    n=('at_risk', 'size')).reindex(['Not-at-risk','Fail','Withdrawn'])
display(summary.round(1))
display(Image(filename=str(FIGS / 'withdrawn_activity_decay.png')))

## 10. Findings & implications for modelling

1. **Early signal (RQ1).** Behavioural/performance features separate the classes from 10–20% of course length and strengthen monotonically; 40–60% is a robust, actionable window.
2. **Behaviour ≫ demographics (RQ1/RQ2).** Engagement/assessment reach *d* > 2; demographic association is small (Cramér's V ≤ 0.15) → SHAP/LIME are expected to rank behavioural features highest.
3. **Mild imbalance (RQ3).** 52.8% at-risk → RQ3 quantifies SMOTE/ADASYN/class-weight against this baseline using PR-AUC/recall.
4. **Correlated features (RQ2).** Two multicollinear pairs may make explanation importance unstable — a factor the stability metric must account for.
5. **No leakage.** No feature is near-perfectly correlated with the label, and the time-aware cut removes future events → held-out estimates should be trustworthy.

**References.** [1] Kuzilek, Hlosta, Zdrahal, *Scientific Data* 4:170171, 2017. [2] Adnan et al., *IEEE Access* 9:7519–7539, 2021. [3] Tomasevic, Gvozdenovic, Vranes, *Computers & Education* 143:103676, 2020.